# Explorer un corpus de presse et comparer des tokenizers

Le corpus de ces travaux pratiques : tous les numéros de 1914 de deux quotidiens, *L'Humanité* (360 numéros) et *Le Figaro* (24 numéros), numérisés par la BnF. Le texte vient d'une reconnaissance de caractères (OCR) brute : il a les défauts d'un vrai corpus d'entreprise.

1. Charger le corpus et mesurer ses statistiques
2. Repérer et corriger les défauts de l'OCR, puis découper en passages
3. Comparer trois tokenizers, et estimer ce que coûte le traitement du corpus

## Données

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s french-press-1914
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
%pip install -q wordfreq

## 1. Charger le corpus

Chaque numéro est un fichier JSON, `french-press-1914/<journal>/1914/<AAAAMMJJ>/<AAAAMMJJ>.metadata.fulltext.json`, dont trois champs nous intéressent :

- `title` : le nom du journal, dans une liste (`content["title"][0]`)
- `date` : la date de parution, dans une liste (`content["date"][0]`)
- `contentAsText` : le texte, une chaîne par page, les lignes séparées par `\n`

Construisez un `pandas.DataFrame` nommé `issues`, une ligne par numéro, avec les colonnes `newspaper`, `date` (au format date) et `text` (les pages jointes par `\n`). Affichez le nombre de numéros par journal et la distribution du nombre de mots par numéro.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path("french-press-1914").rglob("*.json")):
    content = json.loads(path.read_text(encoding="utf8"))
    rows.append((content["title"][0], content["date"][0], "\n".join(content["contentAsText"])))
issues = pd.DataFrame(rows, columns=["newspaper", "date", "text"])
issues["date"] = pd.to_datetime(issues["date"])

print(issues.newspaper.value_counts())
issues.text.str.split().str.len().describe()

Comptez les formes de mots du corpus, en minuscules, avec l'expression régulière `WORD` ci-dessous et un `collections.Counter`. Affichez :

- le nombre total de mots et le nombre de formes différentes
- la part des formes qui n'apparaissent qu'une fois (les *hapax*)
- les 20 formes les plus fréquentes

In [ ]:
import re

WORD = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)?")

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections import Counter

counts = Counter(word.lower() for text in issues.text for word in WORD.findall(text))
total = sum(counts.values())
hapax = sum(1 for count in counts.values() if count == 1)
print(f"{total:,} mots, {len(counts):,} formes, {hapax / len(counts):.0%} de hapax")
counts.most_common(20)

Tracez la fréquence de chaque forme en fonction de son rang, en échelles logarithmiques, avec la droite de la loi de Zipf (fréquence du premier mot divisée par le rang). Les trois quarts des formes sont des hapax : à votre avis, qu'est-ce que c'est ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

frequencies = np.array(sorted(counts.values(), reverse=True))
ranks = np.arange(1, len(frequencies) + 1)
plt.loglog(ranks, frequencies, label="Presse de 1914")
plt.loglog(ranks, frequencies[0] / ranks, "--", label="Loi de Zipf")
plt.xlabel("Rang")
plt.ylabel("Occurrences")
plt.legend()
plt.show()

# Beaucoup de hapax sont des erreurs d'OCR : des mots mal reconnus, coupés ou collés.

## 2. Les défauts de l'OCR

Les mots coupés en fin de ligne sont restés coupés : « con- tinuer ». Comptez les occurrences de l'expression régulière `SPLIT` ci-dessous et affichez-en une dizaine.

In [ ]:
SPLIT = re.compile(r"([^\W\d_]+)-\s+([^\W\d_]+)")

In [ ]:
# Votre code ici

### Solution

In [ ]:
matches = [match for text in issues.text for match in SPLIT.finditer(text)]
print(len(matches))
[match[0] for match in matches[:10]]

Recoller toutes les paires serait une erreur : « Sud- Ouest » doit rester en deux mots. La bibliothèque [`wordfreq`](https://github.com/rspeer/wordfreq) donne la fréquence d'un mot en français : `zipf_frequency(mot, "fr")` vaut 0 pour un mot inconnu.

Écrivez une fonction `rejoin(text)` qui ne recolle une paire que si le mot recollé est connu, puis comptez les paires recollées et celles laissées telles quelles. La fonction `known` mémorise ses réponses : le corpus compte des millions de mots.

In [ ]:
from functools import cache

from wordfreq import zipf_frequency


@cache
def known(word: str) -> bool:
    return zipf_frequency(word.lower(), "fr") > 0

In [ ]:
# Votre code ici

### Solution

In [ ]:
def rejoin(text: str) -> str:
    def fix(match: re.Match[str]) -> str:
        joined = match[1] + match[2]
        return joined if known(joined) else match[0]

    return SPLIT.sub(fix, text)


rejoined = sum(known(match[1] + match[2]) for match in matches)
print(f"{rejoined} paires recollées, {len(matches) - rejoined} laissées telles quelles")
issues["text"] = issues.text.map(rejoin)

Le **taux de bruit** d'un texte : la part de ses mots (de plus d'une lettre) que `wordfreq` ne connaît pas. Écrivez une fonction `noise(text)`, calculez-la pour chaque numéro et comparez les deux journaux. Affichez les 20 formes inconnues les plus fréquentes du corpus : erreurs d'OCR, noms propres, orthographe ancienne ?

Beaucoup sont des moitiés de mots coupés par une espace, sans tiret : « gou vernement », « so cialiste ». La version publiée des passages les recolle aussi, quand l'un des deux morceaux est inconnu et le mot recollé connu.

In [ ]:
# Votre code ici

### Solution

In [ ]:
def words(text: str) -> list[str]:
    return [word for word in WORD.findall(text) if len(word) > 1]


def noise(text: str) -> float:
    found = words(text)
    return sum(not known(word) for word in found) / max(len(found), 1)


issues["noise"] = issues.text.map(noise)
print(issues.groupby("newspaper").noise.describe())
Counter(
    word.lower() for text in issues.text for word in words(text) if not known(word)
).most_common(20)

Un numéro entier est trop long pour un modèle, et mêle des dizaines d'articles : on le découpe en **passages**. Écrivez une fonction `passages(text)` qui accumule les lignes d'un numéro jusqu'à atteindre au moins 100 mots, et coupe à la première ligne qui finit une phrase (par `.`, `!`, `?` ou `»`). Combien de passages obtenez-vous ?

Comparez ensuite avec `passages.parquet`, la version publiée pour les TPs suivants : elle découpe aussi en phrases les lignes de plus de 250 mots, écarte les passages dont le bruit dépasse 0,15 et les doublons. Combien de passages, de quelle longueur ? Lisez-en trois au hasard.

In [ ]:
# Votre code ici

### Solution

In [ ]:
END = re.compile(r"[.!?»]\s*$")


def passages(text: str) -> list[str]:
    out, current, n = [], [], 0
    for line in text.split("\n"):
        line = " ".join(line.split())
        if not line:
            continue
        current.append(line)
        n += len(line.split())
        if n >= 100 and END.search(line):
            out.append(" ".join(current))
            current, n = [], 0
    return out


mine = [passage for text in issues.text for passage in passages(text)]
print(len(mine))

published = pd.read_parquet("passages.parquet")
print(published.newspaper.value_counts())
print(published.n_words.describe())
for text in published.text.sample(3, random_state=0):
    print(text, end="\n\n")

## 3. Comparer des tokenizers

Chargez trois tokenizers avec `transformers.AutoTokenizer.from_pretrained` (le tokenizer seul se télécharge en quelques secondes) :

- `google-bert/bert-base-multilingual-cased` (WordPiece)
- `Qwen/Qwen3-Embedding-0.6B` (BPE sur les octets ; le modèle d'*embeddings* des TPs suivants)
- `FacebookAI/xlm-roberta-base` (Unigram)

Pour chacun, affichez les tokens (décodez chaque identifiant avec `tokenizer.decode([id])` pour qu'ils restent lisibles) et le nombre de tokens par mot des textes de `samples`. Qu'est-ce qui coûte le plus de tokens ?

In [ ]:
samples = {
    "français": "Le conseil municipal a voté hier soir le budget de la nouvelle médiathèque.",
    "anglais": "The city council voted on the budget for the new media library last night.",
    "nom propre": "Clemenceau et Poincaré",
    "OCR": "Où .av b; aifn'wmant Mme Gueydan, it. o ta î - iu ei est arrivée a la barre",
}

In [ ]:
# Votre code ici

### Solution

In [ ]:
from transformers import AutoTokenizer

tokenizers = {
    name: AutoTokenizer.from_pretrained(name)
    for name in (
        "google-bert/bert-base-multilingual-cased",
        "Qwen/Qwen3-Embedding-0.6B",
        "FacebookAI/xlm-roberta-base",
    )
}
for name, tokenizer in tokenizers.items():
    print(name)
    for label, text in samples.items():
        ids = tokenizer(text, add_special_tokens=False).input_ids
        # decode() rend chaque token lisible (Qwen3 découpe des octets, pas des caractères).
        tokens = [tokenizer.decode([token]) for token in ids]
        print(f"  {label:10} {len(ids) / len(text.split()):.2f} tokens/mot {tokens}")

# Le texte bruité de l'OCR coûte le plus : ses morceaux de mots ne sont pas dans les
# vocabulaires. Puis le français, plus cher que l'anglais (vocabulaires appris sur des
# corpus surtout anglais), surtout pour Qwen3.

Combien de tokens (pour le tokenizer de Qwen3) compte le corpus de passages publié ? Estimez-le sur un échantillon de 2 000 passages. À 0,10 € par million de tokens d'entrée (l'ordre de grandeur d'un petit modèle par API), combien coûte un passage du corpus entier dans un LLM, rien qu'en entrée ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
tokenizer = tokenizers["Qwen/Qwen3-Embedding-0.6B"]
sample = published.text.sample(2000, random_state=0)
tokens_per_passage = np.mean([len(tokenizer(text).input_ids) for text in sample])
total_tokens = tokens_per_passage * len(published)
print(f"{tokens_per_passage:.0f} tokens par passage, {total_tokens / 1e6:.1f} millions au total")
print(f"{total_tokens / 1e6 * 0.10:.2f} € pour un passage du corpus")